# STATE -> delta matrix

**One job.** Run Arc's STATE on contexts A, B, C and write

```
state_delta.npz    A, B, C : float32 (300, 18533)   delta in log1p pseudobulk space
                   genes, targets, coverage, checkpoint
```

Nothing else happens here. The blend with retrieval, the harness, the six-member
audit and the packaging all run locally, where the donor corpus and the
instrument already are.

**Enable:** Internet + GPU (T4/P100 is plenty -- inference only).
**Secret:** `VCC_TOKEN` = your challenge API token.
**Optional dataset:** an H1 arm h5ad -- without it there is no validation gate.

## 1 - Environment

In [ ]:
import os, sys, subprocess, glob, json, warnings
warnings.filterwarnings("ignore")

def sh(cmd, check=False):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(((r.stdout or "") + (r.stderr or ""))[-4000:], flush=True)
    if check and r.returncode:
        raise RuntimeError(cmd)
    return r.returncode

sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")

# --- DO NOT DISTURB THE PREINSTALLED NUMPY/SCIPY -------------------------
# Run 2 died here.  `pip install anndata scanpy` pulled numpy 2.5.2, which is
# ABI-incompatible with the scipy already compiled into the Kaggle image, and
# `import anndata` then failed deep inside scipy with
#     ImportError: cannot import name '_center' from 'numpy._core.umath'
# The install warnings said so ("numba requires numpy<2.1, but you have 2.5.2")
# and were ignored.
#
# So: anndata is installed with --no-deps, plus only its own light pure-python
# requirements.  numpy, scipy, pandas and h5py are already present and are left
# strictly alone.  scanpy is dropped entirely -- it was imported and never used.
def ver(mod):
    r = subprocess.run([sys.executable, "-c",
                        f"import {mod};print({mod}.__version__)"],
                       capture_output=True, text=True)
    return r.stdout.strip()

# A CONSTRAINTS FILE, applied to every install.
#
# Run 3 died with numpy 2.0.2 -> 2.5.2 on the FIRST install line: `vcc-cli`
# reads h5ad, so it depends on numpy and dragged it forward.  Guarding only
# anndata with --no-deps was guarding the wrong line.  scipy in the Kaggle image
# is compiled against the preinstalled numpy, so ANY move breaks it with
#     ImportError: cannot import name '_center' from 'numpy._core.umath'
#
# Pinning via -c makes pip either honour the preinstalled versions or fail
# LOUDLY with an unsatisfiable resolution -- both are better than a silent ABI
# break surfacing three cells later.
PINS = {m: ver(m) for m in ("numpy", "scipy", "pandas")}
print("preinstalled:", PINS, flush=True)
with open("/kaggle/working/constraints.txt", "w") as fh:
    for m, v in PINS.items():
        if v:
            print(f"{m}=={v}", file=fh)
CONS = "-c /kaggle/working/constraints.txt"

sh(f"pip -q install {CONS} uv huggingface_hub 2>&1 | tail -3")
sh(f"pip -q install {CONS} vcc-cli 2>&1 | tail -3")
sh(f"pip -q install {CONS} --no-deps anndata natsort scverse_misc array_api_compat "
   f"2>&1 | tail -3")

AFTER = {m: ver(m) for m in PINS}
print("after installs:", AFTER, flush=True)
moved = [m for m in PINS if PINS[m] and PINS[m] != AFTER[m]]
assert not moved, f"{moved} moved: {PINS} -> {AFTER}; scipy ABI will be broken"

sh("uv tool install arc-state 2>&1 | tail -3")

# `uv tool install` writes to ~/.local/bin, which is NOT on PATH in the Kaggle
# image -- the first run installed `state` successfully and then could not run
# it.  Put it on PATH for this process AND every subprocess.
for extra in ("/root/.local/bin", os.path.expanduser("~/.local/bin")):
    if extra not in os.environ["PATH"]:
        os.environ["PATH"] = extra + os.pathsep + os.environ["PATH"]
print("PATH now includes ~/.local/bin", flush=True)

# Hard gate: if the tools are not callable, stop HERE rather than four cells later.
assert sh("state --help 2>&1 | head -5") == 0, "state CLI not runnable"
assert sh("vcc --version 2>&1 | head -3") == 0, "vcc CLI not runnable"
import anndata                  # noqa: F401  -- fail loudly, HERE
import scipy.sparse             # noqa: F401  -- the ABI canary
try:                            # opt in to writing arrow-backed string columns
    anndata.settings.allow_write_nullable_strings = True
except Exception as _e:
    print("  (could not set allow_write_nullable_strings:", _e, ")", flush=True)
print("environment OK", flush=True)

## 2 - Checkpoint

The suffix is the **training modality**, and it decides how much of STATE's
reported advantage actually reaches us:

| checkpoint | trained on | STATE's reported Pearson gain |
|---|---|---|
| `ST-Tahoe` | drugs | +63% |
| `ST-Parse` | PBMC cytokines | +47% |
| **`*-Replogle`** | **genome-wide CRISPRi in cell lines** | **+5%** |

Replogle is our modality, so **+5% is the honest expectation.**

In [ ]:
from huggingface_hub import snapshot_download, list_repo_files

# SE-600M: the EMBEDDING half.  The transition config says `embed_key: X_state`,
# so ST consumes SE embeddings, not counts.  This line was dropped when the
# notebook was rewritten into its delta-only form, and run 5 died on
# `NameError: SE_DIR` in the embedding cell that was added afterwards.
# ---- DOWNLOAD ONLY WHAT IS USED --------------------------------------
# Run 6 filled the Kaggle disk ("high /tmp usage ... Cannot connect to the
# Docker daemon").  Cause: snapshot_download pulls the WHOLE repo.
#   ST-SE-Replogle  195 files  95.12 GB  (3 GB eval h5ads per arm, pure ballast)
#   SE-600M          11 files  29.19 GB  (two 11.55 GB epoch .ckpt)
# = 124 GB into a ~20 GB container.  We need ~5 GB of it.
#
# Keep the HF cache on /kaggle/working, which is the large volume; the default
# ~/.cache lives on the small root disk that just overflowed.
os.environ["HF_HOME"] = "/kaggle/working/hf"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

# SE-600M: model.safetensors (2.86 GB) + protein_embeddings.pt (0.41 GB), which
# `state emb transform` looks for in --model-folder.  The 11.55 GB epoch
# checkpoints are training artefacts and are skipped.
SE_DIR = snapshot_download(
    "arcinstitute/SE-600M",
    allow_patterns=["model.safetensors", "protein_embeddings.pt",
                    "config.yaml", "*.json", "README.md"])
print("SE-600M:", SE_DIR, flush=True)
sh(f"du -sh {SE_DIR}")

# ST-SE-Replogle is organised zeroshot/{hepg2,jurkat,k562,rpe1} and
# fewshot/{...}.  We get ZERO perturbed cells in A/B/C, so the ZEROSHOT arms are
# the ones whose training regime matches our situation -- a fewshot model was
# tuned on examples from the held-out line, which we will never have.
# `st-se-replogle-full` does not expose that distinction at all, so it is only a
# fallback.
CANDIDATES = ["arcinstitute/ST-SE-Replogle", "arcinstitute/st-se-replogle-full",
              "arcinstitute/ST-HVG-Replogle", "arcinstitute/st-x-replogle-full"]
ST_DIR, ST_REPO = None, None
for repo in CANDIDATES:
    try:
        files = list_repo_files(repo)
        # 4 x final.ckpt = 2.04 GB, plus the small per-arm config and onehot
        # maps.  Everything else in the repo -- eval_*/adata_*.h5ad at 3 GB
        # each, best/last checkpoints, per-arm DE csvs -- is never read.
        ST_DIR, ST_REPO = snapshot_download(
            repo,
            allow_patterns=["zeroshot/*/checkpoints/final.ckpt",
                            "zeroshot/*/config.yaml",
                            "zeroshot/*/*.pkl",
                            "*.md"]), repo
        print("USING", repo, "->", files[:10], flush=True)
        break
    except Exception as e:
        print("skip", repo, "|", str(e)[:120], flush=True)
if ST_DIR is None:
    raise RuntimeError("no ST checkpoint resolved")

# The Replogle repo is PER-HELD-OUT-CELL-LINE: hepg2_0.99, jurkat_0.99,
# k562_0.99, rpe1_0.99.  Each model was trained EXCLUDING its named line.  Our
# contexts are none of the four, so every one is zero-shot for us -- but the
# choice still matters, because a checkpoint that INCLUDES jurkat has seen
# lymphoid biology, and context A is lymphoid (marker z +1.66).
#
# For the LANDSCAPE we do better than choosing: we run ALL FOUR and pool their
# predictions.  The basis is estimated from 4 x 300 = 1200 rows instead of 300,
# from models with different held-out lines -- decorrelated evidence, which is
# what a span estimate wants.  Individual per-gene accuracy is discarded anyway;
# only the geometry is kept.
# ZEROSHOT ONLY.  Each arm held out one Replogle cell line and trained on the
# rest, which is exactly our position: A/B/C are lines no model has seen.
# Context A is lymphoid (marker z +1.66) and Jurkat is a T-cell leukaemia line,
# so the arms that INCLUDE jurkat in training have seen the closest biology --
# another reason to pool all four rather than choose one.
CKPTS = sorted(glob.glob(os.path.join(ST_DIR, "zeroshot", "*", "**", "final.ckpt"),
                         recursive=True))
if not CKPTS:
    CKPTS = sorted(glob.glob(os.path.join(ST_DIR, "*", "checkpoints", "final.ckpt")))
if not CKPTS:
    CKPTS = glob.glob(os.path.join(ST_DIR, "**", "*.ckpt"), recursive=True)[:1]
print(f"{len(CKPTS)} checkpoints:")
for c in CKPTS:
    print("   ", os.path.relpath(c, ST_DIR), flush=True)
CKPT = CKPTS[0]

## 3 - Challenge data

`vcc datasets list` advertises one id: `controls` -> `vcc_2026_controls.zip`,
holding one `.h5ad` per context (18,400 non-targeting cells x 18,533 genes)
plus `gene_names.csv` and `pert_counts.csv`. It arrives zipped.

In [ ]:
from kaggle_secrets import UserSecretsClient

# `vcc login` writes to a KEYRING, and a Kaggle container has no keyring
# backend -- run 4 called it, it failed silently (its return code was never
# checked), and the failure surfaced two commands later as
#     vcc: Not logged in. Set VCC_TOKEN, or run `vcc login --token-stdin`
# That message names the fix: the CLI reads a VCC_TOKEN ENVIRONMENT VARIABLE.
# subprocess inherits os.environ, so every `vcc` call below is authenticated
# without touching a keyring.  The value is never printed and never written.
tok = UserSecretsClient().get_secret("VCC_TOKEN")
assert tok, "Kaggle Secret VCC_TOKEN is empty or not attached to this notebook"
os.environ["VCC_TOKEN"] = tok.strip()
print(f"VCC_TOKEN loaded ({len(os.environ['VCC_TOKEN'])} chars)", flush=True)
del tok

# Prove authentication HERE rather than discovering it downstream.
assert sh("vcc whoami 2>&1 | head -8") == 0, "VCC_TOKEN rejected -- token wrong or revoked"

os.makedirs("/kaggle/working/vccdata", exist_ok=True)
assert sh("vcc datasets list") == 0, "datasets list failed"
assert sh("vcc datasets download controls -d /kaggle/working/vccdata") == 0,     "controls download failed"
import zipfile
for z in glob.glob("/kaggle/working/vccdata/*.zip"):
    zipfile.ZipFile(z).extractall("/kaggle/working/vccdata")
sh("find /kaggle/working/vccdata -maxdepth 3 | head -20")

In [ ]:
import anndata as ad, numpy as np, csv

CTRL = {}
for c in "ABC":
    hit = glob.glob(f"/kaggle/working/vccdata/**/context_{c}.h5ad", recursive=True)
    if not hit:
        raise FileNotFoundError(f"context_{c}.h5ad missing")
    CTRL[c] = ad.read_h5ad(hit[0])
    print(c, CTRL[c].shape, flush=True)

GENES = np.asarray(CTRL["A"].var_names, dtype=str)
pcsv = glob.glob("/kaggle/working/vccdata/**/pert_counts.csv", recursive=True)[0]
TARGETS = [r[0] for r in csv.reader(open(pcsv))][1:]
assert len(TARGETS) == 300 and len(GENES) == 18533, (len(TARGETS), len(GENES))
print(len(TARGETS), "targets |", len(GENES), "genes")

## 4a - Embed with SE-600M

The transition model's own config says `embed_key: X_state` -- it was trained on
**SE-600M embeddings**, not raw counts. That is a whole missing stage: Arc ships
TWO pretrained models that chain.

```
control cells (counts) --SE-600M--> X_state --ST-*--> predicted --> genes
```

`state emb transform` writes the embedding into the h5ad. Its `--help` is printed
first because the obsm key name is not documented and we must not guess it.

In [ ]:
sh("state emb transform --help 2>&1 | head -40")

# --model-folder is enough; the checkpoint is resolved inside it.  Listed for
# diagnostics so a missing/renamed weight file is visible.
print("SE files:", [os.path.relpath(p, SE_DIR) for p in
                    sorted(glob.glob(os.path.join(SE_DIR, "*")))][:12], flush=True)

# `--help` shows --embed-key is an INPUT flag: it NAMES the obsm key to write.
# The transition checkpoints were trained with embed_key: X_state, so we set it
# rather than discover it.
EMB_KEY = "X_state"

def embed(in_h5ad, out_h5ad):
    '''Add SE-600M embeddings under EMB_KEY; returns the key, or None.'''
    rc = sh(f"state emb transform --model-folder {SE_DIR} "
            f"--input {in_h5ad} --output {out_h5ad} --embed-key {EMB_KEY}")
    if rc != 0 or not os.path.exists(out_h5ad):
        print("  EMBED FAILED for", in_h5ad, flush=True)
        return None
    import anndata as _ad
    keys = list(_ad.read_h5ad(out_h5ad, backed="r").obsm.keys())
    print("  obsm keys after embed:", keys, flush=True)
    return EMB_KEY if EMB_KEY in keys else (keys[0] if keys else None)

## 4b - Inference

Zero-shot: these contexts contain **no perturbed cells at all**, which is the
whole 2026 task. STATE gets control cells labelled with the target we want
predicted, and returns the predicted profile.

In [ ]:
N_CELLS = int(os.environ.get("N_CELLS", "64"))

def build_input(ctrl, targets, n=N_CELLS, seed=0):
    rng = np.random.default_rng(seed)
    idx, lab = [], []
    for t in targets:
        k = min(n, ctrl.n_obs)
        idx.append(rng.choice(ctrl.n_obs, size=k, replace=False))
        lab += [t] * k
    out = ctrl[np.concatenate(idx)].copy()
    # The control h5ad carries an ARROW-BACKED string index, and slicing keeps
    # that dtype.  anndata refuses to write pd.ArrowStringArray unless opted in
    # (run 7: RuntimeError allow_write_nullable_strings is False).  Cast the
    # index and every object column to plain str, and make the barcodes unique
    # -- the same control cell is sampled for many targets.
    out.obs_names = [f"c{i}" for i in range(out.n_obs)]
    out.var_names = [str(v) for v in out.var_names]
    for _c in list(out.obs.columns):
        out.obs[_c] = out.obs[_c].astype(str)
    out.obs["gene"] = np.asarray(lab, dtype=object)
    out.obs["gene"] = out.obs["gene"].astype("category")
    # the checkpoint config names these explicitly:
    #   pert_col: gene | cell_type_key: cell_line | batch_col: gem_group
    #   control_pert: non-targeting
    out.obs["cell_line"] = "vcc_context"
    out.obs["cell_line"] = out.obs["cell_line"].astype("category")
    out.obs["gem_group"] = "1"
    out.obs["gem_group"] = out.obs["gem_group"].astype("category")
    return out

# Every checkpoint x every context.  Individual per-gene accuracy is not what
# we keep -- the projection uses only the SPAN of these rows -- so more models
# with different held-out lines means a better-conditioned basis.
OUT, EMBKEY = {}, None
for c in "ABC":
    raw = f"/kaggle/working/raw_{c}.h5ad"
    ip = f"/kaggle/working/in_{c}.h5ad"
    build_input(CTRL[c], TARGETS).write_h5ad(raw)
    k = embed(raw, ip)                    # SE-600M -> X_state
    if k is None:
        ip = raw                          # fall through; tx infer will complain
    else:
        EMBKEY = k
    for ck in CKPTS:
        tag = os.path.basename(os.path.dirname(os.path.dirname(ck)))
        op = f"/kaggle/working/out_{c}_{tag}.h5ad"
        mdir = os.path.dirname(os.path.dirname(ck))
        ek = f"--embed-key {EMBKEY} " if EMBKEY else ""
        sh(f"state tx infer --model-dir {mdir} --checkpoint {ck} "
           f"--adata {ip} --pert-col gene {ek}--output {op}")
        OUT[(c, tag)] = op if os.path.exists(op) else None
        print(f"  {c} / {tag} -> {OUT[(c, tag)]}", flush=True)
    for _tmp in (raw, ip):                      # reclaim disk immediately
        if _tmp != ip and os.path.exists(_tmp):
            os.remove(_tmp)
    sh("df -h /kaggle/working | tail -1")

## 5 - Reduce to a delta matrix

Reduced to the space the scorer works in: `log1p(5e4 * pseudobulk)` minus the
control pseudobulk. Coverage is reported per context so a partial failure cannot
pass as a result.

In [ ]:
import scipy.sparse as sp
TS = 5e4

def bulk(X):
    X = X.toarray() if sp.issparse(X) else np.asarray(X)
    P = X.sum(0).astype(np.float64)
    t = P.sum()
    return np.log1p(TS * P / t) if t > 0 else np.zeros_like(P)

def to_delta(pred_path, ctrl, targets):
    a = ad.read_h5ad(pred_path)
    col = next((k for k in ("gene", "target", "pert", "perturbation",
                            "target_gene") if k in a.obs), None)
    if col is None:
        raise KeyError(f"no perturbation column in {list(a.obs.columns)}")
    if a.n_vars != ctrl.n_vars:
        raise ValueError(f"gene dim {a.n_vars} != {ctrl.n_vars} -- STATE may "
                         f"have returned an EMBEDDING, not expression")
    base = bulk(ctrl.X)
    D = np.zeros((len(targets), ctrl.n_vars), dtype=np.float32)
    lab = a.obs[col].astype(str).values
    hit = 0
    for i, t in enumerate(targets):
        m = lab == t
        if m.sum():
            D[i] = (bulk(a.X[m]) - base).astype(np.float32)
            hit += 1
    return D, hit

# Per checkpoint: one delta matrix.  DELTA[c] is the mean across checkpoints
# (a point estimate); SPAN[c] stacks them all, which is what the projection
# basis is actually built from.
DELTA, COV, SPAN = {}, {}, {}
for c in "ABC":
    mats = []
    for ck in CKPTS:
        tag = os.path.basename(os.path.dirname(os.path.dirname(ck)))
        p = OUT.get((c, tag))
        if not p:
            print(f"  {c}/{tag}: NO OUTPUT", flush=True)
            continue
        try:
            D, hit = to_delta(p, CTRL[c], TARGETS)
        except Exception as e:
            print(f"  {c}/{tag}: PARSE FAILED: {str(e)[:100]}", flush=True)
            continue
        mats.append(D)
        print(f"  {c}/{tag}: {hit}/300 | mean |d| "
              f"{float(np.linalg.norm(D, axis=1).mean()):.4f}", flush=True)
    if mats:
        SPAN[c] = np.vstack(mats).astype(np.float32)
        DELTA[c] = np.mean(mats, axis=0).astype(np.float32)
        COV[c] = len(mats)
    else:
        SPAN[c] = np.zeros((1, len(GENES)), np.float32)
        DELTA[c] = np.zeros((300, len(GENES)), np.float32)
        COV[c] = 0
    print(f"{c}: {COV[c]} checkpoints | span rows {SPAN[c].shape[0]}", flush=True)

## 6 - The arm, for the gate

Mount an H1 arm h5ad as a Kaggle dataset and a delta for its targets is emitted
too. That arm has **real measured truth**, so locally we can compute STATE's
cosine against it and compare with our retrieval's **0.16**.

Without this there is no gate -- and a notebook that runs cleanly is not a
notebook that scores.

In [ ]:
ARM = next(iter(glob.glob("/kaggle/input/**/h1*_cells.h5ad", recursive=True)), None)
ARM_D, ARM_T = None, None
if ARM:
    a = ad.read_h5ad(ARM)
    pcol = next(k for k in ("perturbation", "target_gene", "gene") if k in a.obs)
    lab = a.obs[pcol].astype(str).values
    ctrl_arm = a[lab == "non-targeting"].copy()
    ARM_T = sorted(set(lab) - {"non-targeting"})
    print("arm:", ARM, ctrl_arm.shape, len(ARM_T), "targets", flush=True)
    build_input(ctrl_arm, ARM_T).write_h5ad("/kaggle/working/in_arm.h5ad")
    sh(f"state tx infer --model-dir {ST_DIR} --checkpoint {CKPT} "
       f"--adata /kaggle/working/in_arm.h5ad --pert-col gene "
       f"--output /kaggle/working/out_arm.h5ad")
    if os.path.exists("/kaggle/working/out_arm.h5ad"):
        ARM_D, hit = to_delta("/kaggle/working/out_arm.h5ad", ctrl_arm, ARM_T)
        print("arm coverage:", hit, "/", len(ARM_T), flush=True)
else:
    print("arm not mounted -- NO GATE. Do not submit on this alone.", flush=True)

## 7 - Write the matrices

In [ ]:
payload = dict(A=DELTA["A"], B=DELTA["B"], C=DELTA["C"],
               spanA=SPAN["A"], spanB=SPAN["B"], spanC=SPAN["C"],
               genes=GENES, targets=np.array(TARGETS),
               coverage=np.array([COV[c] for c in "ABC"]),
               checkpoint=np.array([ST_REPO or "none"]),
               ckpt_names=np.array([os.path.basename(
                   os.path.dirname(os.path.dirname(c))) for c in CKPTS]))
if ARM_D is not None:
    payload["arm"] = ARM_D
    payload["arm_targets"] = np.array(ARM_T)
np.savez_compressed("/kaggle/working/state_delta.npz", **payload)
print("wrote state_delta.npz",
      round(os.path.getsize("/kaggle/working/state_delta.npz") / 1e6, 1), "MB")
print("coverage A/B/C:", [COV[c] for c in "ABC"], "of 300")
print("checkpoint:", ST_REPO)
print()
print("Download state_delta.npz, then LOCALLY:")
print("  1. statecheck.py  -- STATE's cosine on the arm against our 0.16")
print("  2. blend + harness + official_audit.py --nctrl 18400")
print("  3. build / package / submit ONLY if it clears the six-member gate")